# Ingesta del archivo sprints.json 
1. Leer todos loa archivos desde la carpeta sprints usando API spark dataframe reader 
1. Definir y garantizar el esquema 
1. Agregar las columnas de metadata 
    - Archivo fuente 
    - Ingesta Timestamp 
1. Escribir a la tabla bronze delta 

> Note: JSON is in multi line format

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/02.Ayuda_Bronze

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
# Define source_file and table_name
archivo_fuente = f"{ruta}/sprints"
nombre_tabla = f"{catalogo}.{esquema_bronze}.sprints"

#### 1 - Leer el archivo json usando API dataframe reader

In [0]:
# Definir el esquema
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType

esquema_sprints = StructType([
    StructField("date", StringType()),
    StructField("raceName", StringType()),
    StructField("round", IntegerType()),
    StructField("season", IntegerType()),
    StructField("url", StringType()),
    StructField("constructorId", StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", FloatType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType())
])


In [0]:
# Leer la data de springs
df_sprints = (
    spark.read
       .format('json')
       .schema(esquema_sprints)
       .option('mode', 'FAILFAST')
       .option('multiLine', True)
       .load(archivo_fuente)
)

In [0]:
display(df_sprints)

#### 2 - Agregar las colunmas de metadata
- Archivo Fuente
- Ingesta Timestamp

In [0]:
df_sprints_final = add_ingesta_metadata(df_sprints)

#### 3 - Escribir a la tabla bronze delta

In [0]:
(
    df_sprints_final
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(nombre_tabla)
)

In [0]:
display(spark.table(nombre_tabla))